# Moment Match: late interaction → alignment prior

Event text → train-only BPE → supplied fastText initialization → IDF-weighted token matching → event interaction grid → CNN score + Soft-DTW residual → choose one of four timelines.

**Điểm đã đo ở các version Kaggle trước:** R12 val69.4051%, public.6456/private.6253. Val R04=69.1218%, private.6284. Chênh3/1059 câu val chưa chứng minh tốt hơn ổn định.

**Bài học:** Một vector pooled sớm có thể bỏ mất local match. Cross-attention event-level R11 chưa giúp; Soft-DTW giữ trật tự nhưng vẫn ép match event nhiễu. So R04/R08/R11/R12 để thấy sự khác nhau.

Notebook này chứa source đầy đủ nhưng chưa execute tại bản export này; execution_count=null và outputs rỗng là có chủ ý. Không suy diễn metric mới từ notebook chưa chạy.

Bật GPU T4 trên Kaggle, attach competition tương ứng trước Run All. Loader tự tìm cả `/kaggle/input/competitions/<slug>` và `/kaggle/input/<slug>`. Code assert T4, không tự chuyển CPU. Thực hành sau contest, không dùng trong thời gian cuộc thi cấm trợ giúp.


# Moment Match: so sánh có cấu trúc sau alignment

R04 IDF tốt nhất val (.6912) nhưng Kaggle .6436/.6284. R07 token contextualization + IDF public .6619/private .6162. R08 monotonic prior không vượt R04 val; R10 ghép thêm intact supplied-word channels còn giảm. Thêm feature không bảo đảm giữ được optimization và inductive bias tốt. R10 đổi số kênh CNN nên cũng đổi initialization; không quy hết giảm điểm cho word features.

| Hạng | Paper | Chuyển sang bài này | Không chuyển máy móc |
|---|---|---|---|
| 1 | [A Decomposable Attention Model for NLI, Parikh et al., EMNLP 2016](https://aclanthology.org/D16-1244.pdf) §2 | Attend → compare → aggregate; học phần khác nhau sau khi đã ghép tương ứng | NLI sentence-pair khác chọn timeline; dùng event vectors, không nhận mình tái lập SNLI |
| 2 | [ColBERT, Khattab & Zaharia, SIGIR 2020](https://arxiv.org/html/2004.12832v2) §3 | Giữ token-level late interaction để tránh ép cả câu thành một vector | Không tải BERT; chỉ supplied fastText đã có trong đề. Hard MaxSim R09 đã thử, không mặc định tốt hơn soft |
| 3 | [Text Matching as Image Recognition / MatchPyramid, Pang et al., AAAI 2016](https://arxiv.org/pdf/1602.06359) §MatchPyramid | CNN trên interaction grid học mẫu khớp cục bộ | Ở đây grid event5×9, không cùng grid word bài gốc; giữ nhánh cũ khi thêm reasoning |
| 4 | [Drop-DTW, Dvornik et al., NeurIPS 2021](https://arxiv.org/html/2108.11996v1) §3 | Timeline chứa event không tương ứng; cần cho phép bỏ nhiễu | Chưa cài Drop-DTW đầy đủ. R08 strict subsequence không phải Drop-DTW; nếu full matching gây hại thì đây là nhánh kế |
| 5 | [Soft-DTW, Cuturi & Blondel, ICML 2017](https://proceedings.mlr.press/v70/cuturi17a.html) | Soft-min recurrence cho phép path lặp/căn chỉnh co giãn, gradient truyền qua alignment | Full DTW vẫn ép mọi event phải match; có thể hại nếu nhiều outlier |

**R11** giữ R04 grid branch, thêm nhánh event-level decomposable attention: IDF-weighted pooling BPE → MLP attention → soft alignment 5×9 theo hai chiều → compare [x,aligned,|difference|,product] → aggregate → residual score với scale học được. Đây là cross-attention phụ thuộc candidate, khác self-attention trước khi so sánh ở R03. Không xử lý token như ảnh, không lấy CV weights.

**R12** giữ R04, thêm residual negative Soft-DTW cost trên grid similarity. Recurrence có diagonal/up/left, soft-min gamma .1; khác R08 strict-increasing subsequence. Canary finite gradient và toy all-matched. Mục đích phân biệt bias alignment tốt với ép match nhiễu; không kỳ vọng paper nổi tiếng tự làm điểm tăng.

Cùng seed42/train/provided validation, tối đa30 epoch/240 giây mỗi config. Chọn checkpoint bằng validation. Lưu lỗi tự tin cao để đọc câu target, candidate sai và candidate đúng. Chưa có kết quả GPU vòng 2 khi chốt thiết kế.


In [ ]:
SOURCE_COMMIT = '84e9a9efaa6346578162fa12915a8368f18e6485'
RUN_ID = 12


## Shared utilities

Seed, BPE, masking, data paths, schema checks và report writer.


In [ ]:
"""Utilities for controlled, idea-driven AICC NLP experiments."""
import ast, csv, hashlib, importlib.metadata, json, math, os, random, time
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda:0")


def start_run(task, run_id, config):
    assert torch.cuda.is_available(), "GPU T4 is required; do not silently fall back to CPU"
    name = torch.cuda.get_device_name(0)
    assert "T4" in name, "Expected T4, received " + name
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    info = dict(task=task, run_id=f"R{run_id:02d}", config=config, seed=SEED,
                device=name, gpu_count=torch.cuda.device_count(), started=time.time(),
                source_commit=globals().get("SOURCE_COMMIT"),
                versions={p: importlib.metadata.version(p) for p in
                          ["torch", "numpy", "pandas", "scikit-learn", "tokenizers"]})
    print("RUN_START " + json.dumps(info), flush=True)
    return info


def find_data(slug):
    for root in (Path("/kaggle/input/competitions")/slug, Path("/kaggle/input")/slug):
        if root.is_dir():
            return root
    raise FileNotFoundError("Attach competition data: " + slug)


def make_bpe(texts, size=2048):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    tokenizer.train_from_iterator(texts, trainers.BpeTrainer(
        vocab_size=size, min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return tokenizer


def encode_texts(tokenizer, texts, max_length):
    values = np.zeros((len(texts), max_length), dtype=np.int64)
    for i, encoded in enumerate(tokenizer.encode_batch(texts)):
        ids = encoded.ids[:max_length] or [1]
        values[i, :len(ids)] = ids
    return values


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finish_run(info, validation, submission, checks, history, artifacts=None, limitations=None):
    assert checks.get("ids_unique") is True and checks.get("schema_valid") is True
    assert checks.get("row_count", 0) > 0
    submission.to_csv(OUT/"submission.csv", index=False)
    info.update(status="completed", elapsed_seconds=time.time()-info.pop("started"),
                validation=validation, submission_checks=checks, history=history,
                peak_gpu_memory_mb=torch.cuda.max_memory_allocated()/1024**2,
                artifacts=artifacts or {}, limitations=limitations or [],
                leaderboard_public=None, leaderboard_private=None)
    (OUT/"run_report.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("NLP_RUN_RESULT " + json.dumps(info), flush=True)
    print("NLP_RUN_PASSED", flush=True)
    return info


class PositionEncoder(nn.Module):
    def __init__(self, vocab, dim=96, layers=2, max_length=384, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, dim*3, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids):
        mask = ids.eq(0)
        x = self.embedding(ids) + self.position(torch.arange(ids.shape[1], device=ids.device))
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.norm(x), mask


def pools(hidden, mask):
    valid = (~mask).unsqueeze(-1)
    mean = (hidden*valid).sum(1)/valid.sum(1).clamp(min=1)
    maximum = hidden.masked_fill(~valid, -1e4).amax(1)
    return torch.cat([mean, maximum], -1)



## Model và experiments

Các config Rxx nằm đầu source. Đổi RUN_ID trước khi Run All; mỗi lượt phải bắt đầu model/optimizer mới.


In [ ]:
"""Moment Match: extend official late interaction with BPE and contextual alignment."""
# Reference: AI-Community-Contest/solutions/round-10/moment-match.ipynb (MIT).
from collections import Counter


MOMENT_CONFIGS = {
 1: dict(idea="BPE late interaction plus soft alignment", vocab=2048, token_layers=0, event_context=False, soft=True, temperature=0.2, idf=False, center=False),
 2: dict(idea="Contextualize BPE tokens before late interaction", vocab=2048, token_layers=1, event_context=False, soft=True, temperature=0.2, idf=False, center=False),
 3: dict(idea="Contextualize the event sequence with positional attention", vocab=2048, token_layers=0, event_context=True, soft=True, temperature=0.2, idf=False, center=False),
 4: dict(idea="IDF weighted token matching suppresses common words", vocab=2048, token_layers=0, event_context=False, soft=True, temperature=0.2, idf=True, center=False),
 5: dict(idea="Sharper soft matching tests alignment entropy", vocab=2048, token_layers=0, event_context=False, soft=True, temperature=0.07, idf=False, center=False),
 6: dict(idea="Center-only ablation measures the contribution of neighboring events", vocab=2048, token_layers=0, event_context=False, soft=True, temperature=0.2, idf=False, center=True),
 7: dict(idea="Combine token contextualization and IDF after R03/R04 evidence", vocab=2048, token_layers=1, event_context=False, soft=True, temperature=0.2, idf=True, center=False)
}


# Hypotheses added after the first seven measured experiments.
MOMENT_CONFIGS[8]={**MOMENT_CONFIGS[4],"idea":"IDF plus a differentiable monotonic subsequence alignment prior","monotonic":True}
MOMENT_CONFIGS[9]={**MOMENT_CONFIGS[4],"idea":"Hard max word-piece alignment instead of soft matching","hard":True}


MOMENT_CONFIGS[10]={**MOMENT_CONFIGS[4],"idea":"Fuse scratch BPE matching with intact supplied word-vector matching","word_view":True}
MOMENT_CONFIGS[11]={**MOMENT_CONFIGS[4],"idea":"Event-level decomposable cross attention with compare and aggregate","compare":True}
MOMENT_CONFIGS[12]={**MOMENT_CONFIGS[4],"idea":"Soft-DTW event alignment as a learned residual score","soft_dtw":True}


def soft_dtw_score(similarity,gamma=.1):
    n,m=similarity.shape[-2:]
    zero=similarity.new_zeros(similarity.shape[:-2]);large=zero+1e4
    previous=[zero]+[large]*m
    for i in range(n):
        current=[large]
        for j in range(m):
            costs=torch.stack([previous[j],previous[j+1],current[-1]],-1)
            current.append(1-similarity[...,i,j]-gamma*torch.logsumexp(-costs/gamma,-1))
        previous=current
    return -previous[-1]/(n+m)

def monotonic_alignment(scores,gamma=0.15):
    """Mean score, softly averaged over strictly increasing subsequence paths.

    Not the complete Soft-DTW algorithm: no repeats or horizontal moves. This
    small task-specific prior is inspired by differentiable dynamic programming.
    """
    from math import comb
    n,m=scores.shape[-2:]
    if n>m:raise ValueError("A strict subsequence needs n <= m")
    dp=scores[...,0,:]
    for i in range(1,n):
        values=[]
        for j in range(m):
            if j<i:values.append(scores.new_full(scores.shape[:-2],-1e4))
            else:values.append(scores[...,i,j]+gamma*torch.logsumexp(dp[..., :j]/gamma,dim=-1))
        dp=torch.stack(values,dim=-1)
    log_partition=gamma*torch.logsumexp(dp[...,n-1:]/gamma,dim=-1)
    return (log_partition-gamma*math.log(comb(m,n)))/n


def alignment_canary():
    from itertools import combinations
    toy=(torch.arange(15,device=DEVICE,dtype=torch.float32).reshape(1,1,3,5)%7)/7
    toy.requires_grad_(True);gamma=.15
    got=monotonic_alignment(toy,gamma)
    paths=torch.stack([sum(toy[...,i,j] for i,j in enumerate(path)) for path in combinations(range(5),3)],-1)
    expected=(gamma*torch.logsumexp(paths/gamma,-1)-gamma*math.log(paths.shape[-1]))/3
    assert torch.allclose(got,expected,atol=1e-6)
    got.sum().backward();assert torch.isfinite(toy.grad).all() and (toy.grad>=0).all()
    print("DP_CANARY_PASS "+json.dumps(dict(enumerated_paths=10,value=float(got.detach().cpu().item()))),flush=True)


class MomentMatcher(nn.Module):
    def __init__(self, initial, idf, config, dim=96, word_vectors=None):
        super().__init__()
        self.config = config
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(initial), freeze=False, padding_idx=0)
        self.projection = nn.Linear(initial.shape[1], dim, bias=False)
        self.register_buffer("idf", torch.as_tensor(idf, dtype=torch.float32))
        if config["token_layers"]:
            block = nn.TransformerEncoderLayer(dim, 4, dim*2, dropout=0.1, batch_first=True,
                                               activation="gelu", norm_first=True)
            self.tokens = nn.TransformerEncoder(block, 1, enable_nested_tensor=False)
            self.token_position = nn.Embedding(24, dim)
        if config["event_context"]:
            self.event_position = nn.Embedding(9, dim)
            block = nn.TransformerEncoderLayer(dim, 4, dim*2, dropout=0.1, batch_first=True,
                                               activation="gelu", norm_first=True)
            self.events = nn.TransformerEncoder(block, 1, enable_nested_tensor=False)
            self.context_gate = nn.Parameter(torch.tensor(0.0))
        self.grid = nn.Sequential(nn.Conv2d(4 if config.get("word_view",False) else 2,32,3,padding=1),nn.GELU(),
                                  nn.Conv2d(32,32,3,padding=1),nn.GELU())
        self.head = nn.Sequential(nn.Flatten(),nn.Linear(32*5*9,96),nn.GELU(),
                                  nn.Dropout(0.2),nn.Linear(96,1))
        if config.get('compare'):
            self.attend=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,dim))
            self.compare=nn.Sequential(nn.Linear(dim*4,dim),nn.GELU(),nn.Dropout(.1),nn.Linear(dim,64))
            self.aggregate=nn.Sequential(nn.Linear(128,64),nn.GELU(),nn.Linear(64,1))
            self.compare_scale=nn.Parameter(torch.tensor(.1))
        if config.get('soft_dtw'):self.dtw_scale=nn.Parameter(torch.tensor(.2))

        if config.get("word_view",False):
            self.word_emb=nn.Embedding.from_pretrained(torch.as_tensor(word_vectors),freeze=True,padding_idx=0)
            self.word_projection=nn.Linear(word_vectors.shape[1],dim,bias=False)
        if config.get("monotonic",False):
            self.alignment_scale=nn.Parameter(torch.tensor(.2))

    def representations(self, ids):
        shape=ids.shape
        x=self.projection(self.emb(ids))
        if self.config["token_layers"]:
            z=x.reshape(-1,shape[-1],x.shape[-1])
            positions=self.token_position(torch.arange(shape[-1],device=ids.device))
            z=self.tokens(z+positions,src_key_padding_mask=ids.reshape(-1,shape[-1]).eq(0))
            x=z.reshape(x.shape)
        if self.config["event_context"]:
            mask=ids.ne(0).unsqueeze(-1)
            event=(x*mask).sum(-2)/mask.sum(-2).clamp(min=1)
            e=event.reshape(-1,event.shape[-2],event.shape[-1])
            e=self.events(e+self.event_position(torch.arange(e.shape[1],device=ids.device)))
            x=x+torch.sigmoid(self.context_gate)*e.reshape(event.shape).unsqueeze(-2)
        return F.normalize(x,dim=-1)

    def forward(self, targets,candidates,word_targets=None,word_candidates=None):
        tm,cm=targets.ne(0),candidates.ne(0)
        tv,cv=self.representations(targets),self.representations(candidates)
        sim=torch.einsum("btwd,bcevd->bctewv",tv,cv).float()
        valid=tm[:,None,:,None,:,None]&cm[:,:,None,:,None,:]
        sim=sim.masked_fill(~valid,-1e4)
        temp=self.config["temperature"]
        if self.config.get("hard",False):
            left,right=sim.amax(-1),sim.amax(-2)
        else:
            left=(sim*(sim/temp).softmax(-1)).sum(-1)
            right=(sim*(sim/temp).softmax(-2)).sum(-2)
        tw=self.idf[targets] if self.config["idf"] else tm.float()
        cw=self.idf[candidates] if self.config["idf"] else cm.float()
        tw=tw*tm
        cw=cw*cm
        left=(left*tw[:,None,:,None,:]).sum(-1)/tw[:,None,:,None,:].sum(-1).clamp(min=1e-6)
        right=(right*cw[:,:,None,:,:]).sum(-1)/cw[:,:,None,:,:].sum(-1).clamp(min=1e-6)
        grids=torch.stack([left,right],dim=2)
        if self.config.get("word_view",False):
            wtm,wcm=word_targets.ne(0),word_candidates.ne(0)
            wtv=F.normalize(self.word_projection(self.word_emb(word_targets)),dim=-1)
            wcv=F.normalize(self.word_projection(self.word_emb(word_candidates)),dim=-1)
            ws=torch.einsum("btwd,bcevd->bctewv",wtv,wcv).float()
            wvalid=wtm[:,None,:,None,:,None]&wcm[:,:,None,:,None,:]
            ws=ws.masked_fill(~wvalid,-1e4)
            wl=(ws.amax(-1)*wtm[:,None,:,None,:]).sum(-1)/wtm[:,None,:,None,:].sum(-1).clamp(min=1)
            wr=(ws.amax(-2)*wcm[:,:,None,:,:]).sum(-1)/wcm[:,:,None,:,:].sum(-1).clamp(min=1)
            event_valid=wtm.any(-1)[:,None,:,None]&wcm.any(-1)[:,:,None,:]
            wl=wl.masked_fill(~event_valid,0);wr=wr.masked_fill(~event_valid,0)
            grids=torch.cat([grids,torch.stack([wl,wr],dim=2)],dim=2)
        if self.config["center"]:
            keep=torch.zeros(5,device=targets.device);keep[2]=1
            grids=grids*keep[None,None,None,:,None]
        scores=self.head(self.grid(grids.flatten(0,1))).reshape(-1,4)
        if self.config.get('compare'):
            ta=(tv*tw.unsqueeze(-1)).sum(-2)/tw.sum(-1,keepdim=True).clamp(min=1e-6)
            cb=(cv*cw.unsqueeze(-1)).sum(-2)/cw.sum(-1,keepdim=True).clamp(min=1e-6)
            a=ta[:,None].expand(-1,4,-1,-1);b=cb
            alignment=torch.einsum('bctd,bced->bcte',self.attend(a),self.attend(b))/math.sqrt(a.shape[-1])
            ab=alignment.softmax(-1)@b;ba=alignment.transpose(-1,-2).softmax(-1)@a
            ca=self.compare(torch.cat([a,ab,(a-ab).abs(),a*ab],-1)).mean(-2)
            cc=self.compare(torch.cat([b,ba,(b-ba).abs(),b*ba],-1)).mean(-2)
            scores=scores+self.compare_scale*self.aggregate(torch.cat([ca,cc],-1)).squeeze(-1)
        if self.config.get('soft_dtw'):
            scores=scores+self.dtw_scale*soft_dtw_score((left+right)/2)
        if self.config.get("monotonic",False):
            alignment=monotonic_alignment((left+right)/2)
            scores=scores+self.alignment_scale*alignment
        return scores


def run_moment(run_id):
    config={**MOMENT_CONFIGS[run_id], "epochs":30, "budget_seconds":240, "batch_size":48}
    info=start_run("moment-match",run_id,config)
    if config.get("monotonic",False):alignment_canary()
    if config.get('soft_dtw'):
        toy=torch.eye(3,device=DEVICE)[None].requires_grad_(True)
        val=soft_dtw_score(toy);val.sum().backward()
        assert torch.isfinite(val).all() and torch.isfinite(toy.grad).all()
        assert float(soft_dtw_score(torch.ones(1,2,2,device=DEVICE),gamma=1e-4))>-.001
        print('SOFT_DTW_CANARY_PASS',flush=True)
    root=find_data("moment-match-aicc-round-10")
    texts={s:json.loads((root/f"texts_{s}.json").read_text()) for s in ["train","validation","test"]}
    labels={s:np.load(root/f"labels_{s}.npy") for s in ["train","validation"]}
    events=[line for target,cands in zip(texts["train"]["target"],texts["train"]["candidates"])
            for sequence in [target]+cands for line in sequence]
    tokenizer=make_bpe(events,config["vocab"])
    tokenizer.save(str(OUT/"tokenizer.json"))
    vocab=json.loads((root/"vocab.json").read_text())
    provided=np.load(root/"embedding_matrix.npy")
    initial=np.zeros((tokenizer.get_vocab_size(),provided.shape[1]),dtype=np.float32)
    counts=np.zeros(len(initial),dtype=np.float32)
    for word in sorted(set(w for event in events for w in event.split())):
        if word in vocab:
            for token in tokenizer.encode(word).ids:
                initial[token]+=provided[int(vocab[word])]
                counts[token]+=1
    initial/=np.maximum(counts[:,None],1)
    rng=np.random.default_rng(SEED)
    missing=(counts==0)
    initial[missing]=rng.normal(0,0.02,(int(missing.sum()),provided.shape[1]))
    initial[0]=0
    df=Counter(token for event in events for token in set(tokenizer.encode(event).ids))
    idf=np.array([math.log((len(events)+1)/(df[i]+1))+1 for i in range(len(initial))],dtype=np.float32)
    idf[0]=0
    # The organizer word vocabulary starts at 0, so reserve padding by adding 1.
    word_vectors=np.pad(provided.astype(np.float32),((1,0),(0,0))) if config.get("word_view",False) else None
    def word_ids(sequence):
        result=np.zeros((len(sequence),12),dtype=np.int64)
        for j,event in enumerate(sequence):
            values=[int(vocab[w])+1 for w in event.split() if w in vocab][:12]
            result[j,:len(values)]=values
        return result
    encoded={}
    for split,t in texts.items():
        a=np.stack([encode_texts(tokenizer,seq,24) for seq in t["target"]])
        b=np.stack([np.stack([encode_texts(tokenizer,seq,24) for seq in cands]) for cands in t["candidates"]])
        encoded[split]=(torch.from_numpy(a),torch.from_numpy(b))
        if config.get("word_view",False):
            wa=np.stack([word_ids(seq) for seq in t["target"]])
            wb=np.stack([np.stack([word_ids(seq) for seq in cands]) for cands in t["candidates"]])
            encoded[split]+=torch.from_numpy(wa),torch.from_numpy(wb)
    train_data=TensorDataset(*encoded["train"],torch.as_tensor(labels["train"],dtype=torch.long))
    loader=DataLoader(train_data,batch_size=config["batch_size"],shuffle=True,num_workers=0)
    model=MomentMatcher(initial,idf,config,word_vectors=word_vectors).to(DEVICE)
    if config.get("word_view",False):
        model.eval()
        with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
            canary=[v[:2].to(DEVICE) for v in encoded["validation"]]
            normal=model(*canary)
            canary[2]=torch.zeros_like(canary[2]);canary[3]=torch.zeros_like(canary[3])
            no_word=model(*canary)
            assert torch.isfinite(normal).all() and torch.isfinite(no_word).all()
        print("WORD_VIEW_CANARY_PASS "+json.dumps(dict(word_ids_shifted=1,empty_word_view_finite=True)),flush=True)
    optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=1e-4)
    history=[];best=-1;best_state=None;bad=0;tick=time.time()
    def infer(split):
        model.eval();result=[]
        with torch.inference_mode():
            for batch in DataLoader(TensorDataset(*encoded[split]),batch_size=48):
                with torch.autocast("cuda",dtype=torch.float16):
                    scores=model(*[value.to(DEVICE) for value in batch])
                assert torch.isfinite(scores).all()
                result.append(scores.float().cpu())
        return torch.cat(result)
    scaler=torch.amp.GradScaler("cuda")
    for epoch in range(config["epochs"]):
        model.train();losses=[]
        for *features,y in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=torch.float16):
                scores=model(*[value.to(DEVICE) for value in features]);loss=F.cross_entropy(scores,y.to(DEVICE))
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),1)
            scaler.step(optimizer);scaler.update();losses.append(float(loss.detach()))
        scores=infer("validation")
        acc=float((scores.argmax(1).numpy()==labels["validation"]).mean())
        row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),validation_accuracy=acc)
        history.append(row);print("EPOCH "+json.dumps(row),flush=True)
        if acc>best:
            best=acc;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};bad=0
        else:bad+=1
        if bad>=7 or time.time()-tick>config["budget_seconds"]:break
    model.load_state_dict(best_state)
    val_scores=infer("validation").numpy();test_scores=infer("test").numpy()
    assert np.isfinite(test_scores).all()
    np.save(OUT/"validation_logits.npy",val_scores)
    np.save(OUT/"validation_labels.npy",labels["validation"])
    probability=torch.softmax(torch.from_numpy(val_scores),1).numpy()
    predicted=val_scores.argmax(1);gold=labels["validation"]
    ordered=np.sort(probability,axis=1);margins=ordered[:,-1]-ordered[:,-2]
    wrong=np.flatnonzero(predicted!=gold)
    selected=wrong[np.argsort(-probability[wrong,predicted[wrong]])[:8]]
    cases=[dict(row_id=int(i),gold=int(gold[i]),prediction=int(predicted[i]),confidence=float(probability[i,predicted[i]]),
                target=texts["validation"]["target"][i],
                chosen=texts["validation"]["candidates"][i][predicted[i]],
                expected=texts["validation"]["candidates"][i][gold[i]]) for i in selected]
    (OUT/"validation_errors.json").write_text(json.dumps(cases,ensure_ascii=False,indent=2))
    print("VALIDATION_ERRORS "+json.dumps(cases,ensure_ascii=False),flush=True)
    print("VALIDATION_DIAGNOSTICS "+json.dumps(dict(mean_margin=float(margins.mean()),wrong_count=int(len(wrong)))),flush=True)
    torch.save(best_state,OUT/"model.pt")
    sub=pd.DataFrame(dict(row_id=np.arange(len(test_scores)),label=test_scores.argmax(1)))
    checks=dict(ids_unique=sub.row_id.is_unique,schema_valid=list(sub.columns)==["row_id","label"],
                row_count=len(sub),labels_valid=bool(sub.label.between(0,3).all()))
    return finish_run(info,dict(metric="accuracy",value=best,direction="maximize",
          samples=len(labels["validation"]),split="provided_validation"),sub,checks,history,
          artifacts=dict(tokenizer="tokenizer.json",weights="provided_fastText_only",word_feature_view=bool(config.get("word_view",False)),
                         source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if "__file__" in globals() and Path(__file__).exists() else None),
          limitations=["BPE vocabulary trained on train text only; supplied fastText initializes its vectors.",
                       "Reference architecture is adapted, not claimed to reproduce its 0.7243 result."])



In [ ]:
OUT = Path('/kaggle/working') / 'moment-match' / f'R{RUN_ID:02d}'
OUT.mkdir(parents=True, exist_ok=True)
report = run_moment(RUN_ID)
print('Submission:', OUT / 'submission.csv')
